# 🎯  실습: OutputParser와 Pydantic

 `chain = prompt | llm` → 자연어 답변
 `chain = prompt | llm | parser` → 정형 데이터 답변

체인 끝에 **파서 하나만** 추가됩니다. 

## 📋 빌드업 흐름

Day 1 패턴에서 부품이 하나씩 추가됩니다.

```
[출발점] Day 1 구조
   chain = prompt | llm                     
        ↓ Step 2 자연어 답변의 한계 직접 체험
[추가 ①] 출력을 명시적으로 받기
   chain = prompt | llm | StrOutputParser()  ← 첫 파서 추가
        ↓ Step 3 Pydantic으로 데이터 구조 정의
[추가 ②] 정형 데이터로 받기
   chain = prompt | llm | PydanticOutputParser  ← 객체로 받음 ⭐
        ↓ Step 4 같은 패턴을 다양한 사례에 적용
[응용] 5가지 실무 미니 케이스
   - 리뷰 분석 / 문의 라우팅 / 이력서 추출 /
     뉴스 태깅 / 이메일 우선순위
        ↓ Step 5 여러 건 한 번에 처리
[확장] chain.batch()
```

## 📋 노트북 순서

```
Step 0. 환경 설정
Step 1. Day 1 패턴 복습              ← prompt | llm
Step 2. 자연어 답변의 한계
Step 3. ① StrOutputParser 추가         ← 첫 번째 부품
Step 4. ② Pydantic + 파서 추가 ⭐     ← 두 번째 부품
Step 5. 5가지 실무 케이스로 응용
Step 6. batch로 여러 건 동시 처리
Step 7. 도전 과제
```


---

# Step 0. 환경 설정


In [1]:
# 패키지 설치 
#!uv add  langchain langchain-openai python-dotenv pydantic

---

# Step 1. Day 1 패턴 복습 — `prompt | llm`

오늘 새로운 걸 배우기 전에 **어제 익숙해진 LCEL 구조**를 한 번 더 손에 익힙니다. 출발점이 명확해야 새로 추가되는 부품의 역할이 보여요.

## 1-1. Day 1에서 항상 쓰던 import


In [1]:
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate

# 분류·추출 작업은 일관성이 중요 → 항상 temperature=0
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

print("✅부품 준비 완료")

✅부품 준비 완료


## 1-2.LCEL 패턴 — `prompt | llm`



In [2]:
# Day 1 패턴 그대로
prompt = ChatPromptTemplate.from_messages([
    ("system", "당신은 친절한 리뷰 분석가입니다. 한국어로 답하세요."),
    ("human", "이 리뷰를 분석해줘: {review}")
])

# 체인 조립
chain_v1 = prompt | llm

# 실행
result = chain_v1.invoke({"review": "이 카메라 진짜 좋아요! 가격도 싸고 화질도 최고!"})
print(result.content)

이 리뷰는 긍정적인 감정을 담고 있습니다. 사용자는 카메라에 대해 매우 만족하고 있으며, 두 가지 주요 포인트를 강조하고 있습니다. 

1. **가격**: "가격도 싸고"라는 표현은 이 카메라가 가성비가 뛰어난 제품임을 나타냅니다. 소비자들은 가격이 저렴하면서도 품질이 좋은 제품을 선호하기 때문에, 이 점은 구매를 고려하는 다른 소비자들에게 매력적으로 다가올 수 있습니다.

2. **화질**: "화질도 최고!"라는 부분은 카메라의 성능에 대한 높은 평가를 보여줍니다. 화질은 카메라 선택에서 가장 중요한 요소 중 하나이므로, 이 리뷰는 제품의 핵심 장점을 잘 전달하고 있습니다.

전반적으로 이 리뷰는 카메라의 가격과 화질에 대한 긍정적인 경험을 공유하고 있으며, 다른 소비자들에게도 추천할 만한 제품이라는 인상을 줍니다.


**✅ 결과 확인**

답변이 잘 나옵니다. 친절하고 자세하게요. **하지만 한 가지 문제**가 있어요. 다음 단계에서 그 문제를 직접 느껴봅시다.


---

# Step 2. 자연어 답변의 한계 직접 체험

같은 체인으로 **같은 종류의 작업**을 3번 반복해봅시다.


In [5]:
# 비슷한 리뷰 3개를 같은 체인으로 분석
reviews = [
    "이 카메라 진짜 좋아요! 가격도 싸고 화질도 최고!",
    "배송이 너무 느려요. 일주일 넘게 걸렸어요.",
    "디자인은 마음에 드는데 가격이 좀 비싸네요.",
]

for r in reviews:
    res = chain_v1.invoke({"review": r})
    print(f"📝 {r}")
    print(f"💬 {res.content[:80]}...")
    print()

📝 이 카메라 진짜 좋아요! 가격도 싸고 화질도 최고!
💬 이 리뷰는 긍정적인 감정을 담고 있습니다. 사용자는 카메라에 대해 매우 만족하고 있으며, 두 가지 주요 포인트를 강조하고 있습니다. 

1. *...

📝 배송이 너무 느려요. 일주일 넘게 걸렸어요.
💬 이 리뷰는 배송 속도에 대한 불만을 표현하고 있습니다. 사용자는 상품이 도착하는 데 일주일 이상 걸렸다고 언급하며, 이는 고객의 기대에 미치지 ...

📝 디자인은 마음에 드는데 가격이 좀 비싸네요.
💬 이 리뷰는 제품이나 서비스에 대한 긍정적인 요소와 부정적인 요소를 모두 포함하고 있습니다. 

1. **긍정적인 요소**: "디자인은 마음에 드...



**👀 무엇이 문제일까?**

답변은 잘 나옵니다. 하지만:

| 우리가 원하는 것 | 실제로 받은 것 |
| --- | --- |
| `rating = 5` (정수) | "긍정적인 리뷰입니다..." (자연어) |
| `sentiment = "positive"` | "고객님께서 만족하셨고..." |
| `keywords = ["카메라", "화질"]` | "카메라의 화질과 가격에 대한 만족감..." |

답변 형식이 **매번 조금씩 다르고**, 평점·감정·키워드를 **텍스트에서 일일이 뽑아내야** 합니다. 리뷰 100건이면 100번 파싱. 불가능에 가까워요.

→ 이걸 해결하는 게 **OutputParser**입니다.


---

# Step 3. ① `StrOutputParser` 추가 — 첫 번째 파서

본격적인 정형화 전에, **출력을 명시적으로 받는 가장 단순한 파서**부터 만나봅시다.

## 3-1. StrOutputParser는 뭐가 다른가?

지금 `chain_v1.invoke(...)` 결과는 `AIMessage` 객체였어요. 그래서 `.content` 로 텍스트를 꺼내야 했죠.

`StrOutputParser` 를 체인 끝에 붙이면 **바로 문자열**이 나옵니다.


In [6]:
from langchain_core.output_parsers import StrOutputParser

# Day 1 체인에 파서 하나만 추가
chain_v2 = prompt | llm | StrOutputParser()
#                         ↑↑↑↑↑↑↑↑↑↑↑↑↑↑↑ 추가된 부품

# 실행
result = chain_v2.invoke({"review": "이 카메라 진짜 좋아요!"})

print(f"타입: {type(result).__name__}")
print(f"값: {result}")

타입: TextAccessor
값: 이 리뷰는 긍정적인 감정을 표현하고 있습니다. "진짜 좋아요!"라는 표현은 사용자가 카메라에 대해 매우 만족하고 있다는 것을 나타냅니다. 리뷰가 간결하지만, 강한 긍정적인 인상을 주며, 제품의 품질이나 성능에 대한 신뢰를 암시합니다. 다만, 구체적인 기능이나 사용 경험에 대한 자세한 설명이 없기 때문에, 다른 소비자들이 참고하기에는 정보가 부족할 수 있습니다. 추가적인 세부사항이나 사용 사례가 포함되면 더욱 유용한 리뷰가 될 것입니다.


**비교**

| 체인 | 출력 타입 | 텍스트 꺼내기 |
| --- | --- | --- |
| `prompt \| llm` | `AIMessage` | `result.content` 필요 |
| `prompt \| llm \| StrOutputParser()` | `str` | 바로 사용 가능 |

차이가 크진 않아요. 하지만 **"체인에 파서를 붙인다"** 는 개념이 손에 익으면 다음 단계가 자연스러워집니다.

진짜 강력한 건 다음에 추가할 **PydanticOutputParser** 입니다.


---

# Step 4. ② `PydanticOutputParser` 추가 — 정형 데이터로! ⭐

이제 본격적입니다. 출력을 **Pydantic 객체**로 받습니다.

## 4-1. 먼저 Pydantic 친해지기 (LLM 없이)

LLM과 결합하기 전에 Pydantic만 단독으로 봅시다. 이게 뭐 하는 도구인지 알면 다음이 훨씬 쉬워요.


In [7]:
from pydantic import BaseModel, Field
from typing import Literal, Optional

# 가장 단순한 Pydantic 클래스
class Person(BaseModel):
    name: str
    age: int

# 정상 생성 - 일반 클래스처럼 사용
p1 = Person(name="윤미", age=30)
print(f"이름: {p1.name}, 나이: {p1.age}")

이름: 윤미, 나이: 30


In [8]:
# 잘못된 타입을 넣으면 자동으로 에러
try:
    p2 = Person(name="윤미", age="삼십")
except Exception as e:
    print(f"❌ {type(e).__name__}: 'age'는 정수여야 합니다")

❌ ValidationError: 'age'는 정수여야 합니다


## 4-2. Field + Literal로 더 정밀한 스키마

`Field` 로 설명을, `Literal` 로 허용값을 제한할 수 있어요 (이론 2부 문법 3·4).


In [9]:
# 리뷰 분석 결과를 표현하는 스키마
class ReviewAnalysis(BaseModel):
    rating: int = Field(description="평점 1-5점", ge=1, le=5)
    sentiment: Literal["positive", "negative", "neutral"] = Field(description="감정")
    keywords: list[str] = Field(description="핵심 키워드 3개")
    summary: str = Field(description="한 문장 요약")

# 정상 생성
r = ReviewAnalysis(rating=5, sentiment="positive",
                   keywords=["카메라", "화질"], summary="좋은 카메라")
print(r)

rating=5 sentiment='positive' keywords=['카메라', '화질'] summary='좋은 카메라'


## 4-3. 이 스키마를 LLM과 연결 — 핵심 5단계 패턴

이제 이 Pydantic 클래스를 **LLM의 출력 형식**으로 사용합니다. 이론 4부의 5단계 패턴 그대로.


In [11]:
from langchain_core.output_parsers import PydanticOutputParser

# 1단계: 스키마 → (위에서 ReviewAnalysis로 정의 완료)
# 2단계: 파서 만들기
parser = PydanticOutputParser(pydantic_object=ReviewAnalysis)

# 3단계: 프롬프트 — {format_instructions} 변수에 파서 지시문 자동 삽입
prompt_v3 = ChatPromptTemplate.from_messages([
    ("system",
     "당신은 리뷰 분석가입니다. 모든 텍스트는 한국어로 작성하세요.\n\n"
     "{format_instructions}"),
    ("human", "리뷰: {review}")
]).partial(format_instructions=parser.get_format_instructions())        

# 4단계: LLM (Step 0에서 이미 준비 — temperature=0)
# 5단계: 체인 조립
chain_v3 = prompt_v3 | llm | parser
#                            ↑↑↑↑↑↑ Day 1 체인에 비해 추가된 부품

print("✅ 5단계 패턴 체인 완성")

✅ 5단계 패턴 체인 완성


## 4-4. 실행 — 자연어가 객체로!

Step 2와 똑같은 리뷰를 입력해서 결과를 비교해봅시다.


In [12]:
# Step 2와 똑같은 리뷰
review = "이 카메라 진짜 좋아요! 가격도 싸고 화질도 최고!"

result = chain_v3.invoke({"review": review})

# 이제 객체 속성으로 접근!
print(f"타입:     {type(result).__name__}")
print(f"⭐ 평점:  {result.rating}/5")
print(f"😊 감정:  {result.sentiment}")
print(f"🏷️  키워드: {', '.join(result.keywords)}")
print(f"📝 요약:  {result.summary}")

타입:     ReviewAnalysis
⭐ 평점:  5/5
😊 감정:  positive
🏷️  키워드: 카메라, 가격, 화질
📝 요약:  이 카메라는 가격이 저렴하고 화질이 뛰어납니다.


**🎉 첫 객체 변환 성공!**

Step 2에서는 자연어로 받아서 텍스트를 일일이 파싱해야 했는데, 이제는:

```python
result.rating        # 5 (정수, DB 저장 가능)
result.sentiment     # "positive" (필터링 가능)
result.keywords      # ["카메라", "가격", "화질"] (검색 인덱싱 가능)
```

체인의 변화를 다시 확인해보세요.

```python
# Step 1 (Day 1)        prompt | llm                          → AIMessage
# Step 3 (추가 ①)       prompt | llm | StrOutputParser()      → str
# Step 4 (추가 ②) ⭐    prompt | llm | PydanticOutputParser   → 객체
```

**파서 하나만 바꿨을 뿐**인데 LLM의 활용 가능성이 완전히 달라졌습니다.


---

# Step 5. ⭐ 5가지 실무 미니 케이스

같은 5단계 패턴을 **5가지 다른 실무 상황**에 적용해봅시다.

**케이스마다 바뀌는 건 3가지**:
1. **스키마** (Pydantic 클래스)
2. **프롬프트** (system 메시지)
3. **입력 텍스트**

체인 조립(`prompt | llm | parser`)은 매번 똑같아요. 이 반복을 통해 패턴이 자연스럽게 손에 익습니다.


## 5-1. 리뷰 분석 확장 — 카테고리 자동 분류

Step 4 분석기에 **카테고리** 와 **개선 제안** 필드를 추가합니다.


In [13]:
# 스키마 — 카테고리 + 개선 제안 추가
class ReviewFull(BaseModel):
    rating: int = Field(description="평점 1-5점", ge=1, le=5)
    sentiment: Literal["positive", "negative", "neutral"] = Field(description="감정")
    category: Literal["품질", "가격", "배송", "디자인", "기타"] = Field(description="리뷰가 다루는 영역")
    keywords: list[str] = Field(description="핵심 키워드 3개")
    improvement: Optional[str] = Field(default=None, description="개선 제안 (있으면, 없으면 None)")

parser_r = PydanticOutputParser(pydantic_object=ReviewFull)
prompt_r = ChatPromptTemplate.from_messages([
    ("system", "쇼핑몰 리뷰 분석가입니다. 한국어로 답하세요.\n\n{format_instructions}"),
    ("human", "리뷰: {review}")
]).partial(format_instructions=parser_r.get_format_instructions())

chain_review = prompt_r | llm | parser_r

In [14]:
# 테스트 — 3개 리뷰
reviews = [
    "포장이 너무 아쉽네요. 박스가 찌그러져 왔어요.",
    "가성비 짱! 이 가격에 이 정도면 최고예요.",
    "디자인은 예쁜데 기능이 좀 부족하네요.",
]

for r in reviews:
    res = chain_review.invoke({"review": r})
    print(f"📝 '{r}'")
    print(f"   ⭐ {res.rating}/5 | {res.sentiment} | 📁 {res.category}")
    print(f"   🏷️  {', '.join(res.keywords)}")
    if res.improvement:
        print(f"   💡 개선: {res.improvement}")
    print()

📝 '포장이 너무 아쉽네요. 박스가 찌그러져 왔어요.'
   ⭐ 3/5 | negative | 📁 기타
   🏷️  포장, 박스, 찌그러짐
   💡 개선: 포장 상태 개선이 필요합니다.

📝 '가성비 짱! 이 가격에 이 정도면 최고예요.'
   ⭐ 5/5 | positive | 📁 가격
   🏷️  가성비, 최고, 가격

📝 '디자인은 예쁜데 기능이 좀 부족하네요.'
   ⭐ 3/5 | neutral | 📁 디자인
   🏷️  디자인, 기능, 부족
   💡 개선: 기능 개선이 필요합니다.



**👀 관찰 포인트**

- 부정 리뷰는 자동으로 `category="배송"` 같이 적절히 분류
- `Optional[str]` 로 만든 `improvement` 필드는 **있을 때만** 채워짐
- `Literal` 로 카테고리를 제한했기 때문에 LLM이 마음대로 새 카테고리를 만들지 못함


## 5-2. 고객 문의 자동 라우팅

고객센터 문의를 **적절한 팀에 자동 배정**하는 시스템 (이론 5부 사례 2).


In [15]:
# 스키마 — 카테고리 + 긴급도 + 담당팀
class InquiryRouting(BaseModel):
    category: Literal["배송", "결제", "환불", "상품문의", "기타"] = Field(description="문의 카테고리")
    urgency: Literal["low", "medium", "high"] = Field(description="긴급도")
    team: Literal["물류팀", "결제팀", "CS팀", "고객지원팀"] = Field(description="담당 팀")
    summary: str = Field(description="문의 한 줄 요약")

parser_i = PydanticOutputParser(pydantic_object=InquiryRouting)
prompt_i = ChatPromptTemplate.from_messages([
    ("system",
     "고객 문의를 적절한 팀에 라우팅합니다.\n"
     "환불 요청·결제 거절·격앙된 톤은 high, 단순 문의는 low로 분류하세요.\n\n"
     "{format_instructions}"),
    ("human", "문의: {inquiry}")
]).partial(format_instructions=parser_i.get_format_instructions())

chain_inquiry = prompt_i | llm | parser_i

In [16]:
# 테스트 — 5건 문의
inquiries = [
    "어제 주문한 게 아직 안 와요. 빨리 처리해주세요.",
    "카드 결제가 자꾸 거절되네요.",
    "받은 상품이 불량입니다! 당장 환불해주세요!!",
    "이 제품 색상이 사진이랑 다른가요?",
    "주문 취소하고 싶어요.",
]

for q in inquiries:
    res = chain_inquiry.invoke({"inquiry": q})
    icon = {"high": "🚨", "medium": "⚠️", "low": "📌"}[res.urgency]
    print(f"{icon} [{res.category}] → {res.team}")
    print(f"   '{q}'")
    print(f"   요약: {res.summary}")
    print()

⚠️ [배송] → 물류팀
   '어제 주문한 게 아직 안 와요. 빨리 처리해주세요.'
   요약: 어제 주문한 상품 배송 지연 문의

🚨 [결제] → 결제팀
   '카드 결제가 자꾸 거절되네요.'
   요약: 카드 결제가 자꾸 거절됨

🚨 [환불] → CS팀
   '받은 상품이 불량입니다! 당장 환불해주세요!!'
   요약: 받은 상품이 불량으로 환불 요청

📌 [상품문의] → 고객지원팀
   '이 제품 색상이 사진이랑 다른가요?'
   요약: 제품 색상에 대한 문의

🚨 [결제] → 결제팀
   '주문 취소하고 싶어요.'
   요약: 주문 취소 요청



**👀 관찰 포인트**

- "당장!!" 같은 강한 톤 → 자동으로 `high` 우선순위
- LLM이 카테고리·긴급도·담당팀을 **동시에** 결정

실제 콜센터·이커머스가 매일 수만 건 처리하는 핵심 패턴입니다.


## 5-3. 이력서 자동 정보 추출

자연어 이력서에서 **DB 저장용 정형 데이터**를 추출 (이론 5부 사례 3).

여기서는 `Optional` 을 활용해 **있을 수도, 없을 수도 있는 필드**를 표현합니다.


In [17]:
# 스키마 — Optional 필드 활용
class ResumeInfo(BaseModel):
    name: str = Field(description="이름")
    years_of_experience: int = Field(description="총 경력 연수")
    skills: list[str] = Field(description="기술 스택")
    education: str = Field(description="최종 학력")
    preferred_role: str = Field(description="희망 직무")
    expected_salary: Optional[int] = Field(default=None, description="희망 연봉(만원). 명시 없으면 None")

parser_re = PydanticOutputParser(pydantic_object=ResumeInfo)
prompt_re = ChatPromptTemplate.from_messages([
    ("system", "이력서에서 정보를 추출합니다. 명시되지 않은 항목은 None으로 두세요.\n\n{format_instructions}"),
    ("human", "이력서:\n{resume}")
]).partial(format_instructions=parser_re.get_format_instructions())

chain_resume = prompt_re | llm | parser_re

In [18]:
# 테스트 — 자연어 이력서
resume_text = """
안녕하세요. 저는 김진환이라고 합니다.
서울대학교 통계학과를 졸업하고 데이터 분석가로 8년째 일하고 있어요.
주로 Python, SQL, Pandas, PyTorch를 사용하며 최근에는 LangChain으로
LLM 애플리케이션도 만들고 있습니다.
다음 직장은 시니어 데이터 사이언티스트로 가고 싶고, 연봉은 7,500만원 정도 희망합니다.
"""

res = chain_resume.invoke({"resume": resume_text})
print(f"👤 이름: {res.name}")
print(f"💼 경력: {res.years_of_experience}년")
print(f"🛠  기술: {', '.join(res.skills)}")
print(f"🎓 학력: {res.education}")
print(f"🎯 희망 직무: {res.preferred_role}")
print(f"💰 희망 연봉: {res.expected_salary}만원")

👤 이름: 김진환
💼 경력: 8년
🛠  기술: Python, SQL, Pandas, PyTorch, LangChain
🎓 학력: 서울대학교 통계학과
🎯 희망 직무: 시니어 데이터 사이언티스트
💰 희망 연봉: 7500만원


**👀 관찰 포인트**

- 자연어로 쓰인 이력서가 **DB에 바로 저장 가능한 형태**로 변환
- `list[str]` 로 기술 스택이 자동 리스트화 → 검색·필터링 가능

HR 시스템에서 매우 자주 쓰이는 패턴이에요.

> 💡 **시도해보기**: 위 이력서 텍스트에서 "연봉은 7,500만원" 부분을 지우고 다시 실행해보세요. `expected_salary` 가 `None` 으로 나옵니다.


## 5-4. 뉴스 기사 다중 태깅

뉴스 기사에 **여러 개의 카테고리 태그**를 자동 부여하는 사례.

이전 케이스들은 카테고리를 하나만 골랐지만, 여기서는 **`list[Literal[...]]`** 로 여러 개를 동시에 선택합니다.


In [19]:
# 스키마 — 다중 태그
class NewsAnalysis(BaseModel):
    title: str = Field(description="기사의 짧은 제목 (15자 이내)")
    topics: list[Literal["정치", "경제", "사회", "기술", "문화", "스포츠", "건강", "환경"]] = Field(
        description="해당하는 모든 주제 태그 (복수 선택)"
    )
    importance: Literal[1, 2, 3, 4, 5] = Field(description="중요도 1-5")
    one_line: str = Field(description="기사 한 줄 요약")

parser_n = PydanticOutputParser(pydantic_object=NewsAnalysis)
prompt_n = ChatPromptTemplate.from_messages([
    ("system", "뉴스 기사를 분석하고 적절한 태그를 부여합니다. 한국어로 답하세요.\n\n{format_instructions}"),
    ("human", "기사:\n{article}")
]).partial(format_instructions=parser_n.get_format_instructions())

chain_news = prompt_n | llm | parser_n

In [20]:
# 테스트 — 복합 주제의 뉴스
articles = [
    "정부가 AI 산업 육성을 위해 5조 원 규모 지원금을 편성했다. "
    "한국의 글로벌 기술 경쟁력 강화를 위한 정책으로, 관련 스타트업과 연구 기관들이 수혜를 받게 된다.",

    "프로야구 한국시리즈에서 LG 트윈스가 우승을 차지했다. "
    "30년 만의 우승에 시민들의 환호가 이어졌다.",

    "전 세계적으로 폭염이 지속되며 농작물 피해가 심각하다. "
    "농가는 정부 지원을 호소하고 있고, 식료품 가격 인상도 우려된다.",
]

for a in articles:
    res = chain_news.invoke({"article": a})
    stars = "⭐" * res.importance
    print(f"📰 {res.title}  {stars}")
    print(f"   태그: {', '.join(res.topics)}")
    print(f"   요약: {res.one_line}")
    print()

📰 AI 산업 지원금 5조  ⭐⭐⭐⭐
   태그: 경제, 기술
   요약: 정부가 AI 산업 육성을 위해 5조 원 지원금을 편성했다.

📰 LG 트윈스, 30년 만에 우승  ⭐⭐⭐⭐
   태그: 스포츠
   요약: LG 트윈스가 30년 만에 한국시리즈에서 우승했다.

📰 폭염으로 농작물 피해  ⭐⭐⭐⭐
   태그: 사회, 경제, 환경
   요약: 폭염으로 인한 농작물 피해와 식료품 가격 인상 우려.



**👀 관찰 포인트**

- 첫 기사는 `["정치", "경제", "기술"]` 같이 **여러 태그 동시 부여**
- 폭염 기사는 `["환경", "사회", "경제"]` 등 복합 주제 인식
- `list[Literal[...]]` 패턴의 강력함

검색엔진 자동 분류, 추천 시스템의 컨텐츠 태깅에 자주 사용됩니다.


## 5-5. 이메일 우선순위 자동 판단

매일 들어오는 이메일을 **자동 분류**하고 **답장 필요 여부**를 판단 (이론 5부 사례 4).

여기서는 **`bool` 필드**로 "답장 필요? Yes/No" 를 받습니다.


In [21]:
# 스키마 — bool 필드 활용
class EmailTriage(BaseModel):
    category: Literal["업무", "개인", "광고", "스팸"] = Field(description="이메일 카테고리")
    priority: Literal["low", "medium", "high"] = Field(description="우선순위")
    requires_reply: bool = Field(description="답장이 필요한 메일인지")
    suggested_action: Literal["즉시 답장", "오늘 중 답장", "보관", "삭제"] = Field(description="추천 액션")
    one_line: str = Field(description="한 줄 요약")

parser_e = PydanticOutputParser(pydantic_object=EmailTriage)
prompt_e = ChatPromptTemplate.from_messages([
    ("system",
     "이메일을 분류하고 처리 방법을 제안합니다.\n"
     "광고·스팸은 답장 불필요, 업무 메일은 보통 답장 필요로 분류하세요.\n\n"
     "{format_instructions}"),
    ("human", "제목: {subject}\n본문: {body}")
]).partial(format_instructions=parser_e.get_format_instructions())

chain_email = prompt_e | llm | parser_e

In [22]:
# 테스트 — 다양한 이메일
emails = [
    {"subject": "[긴급] 내일 오전 회의 자료 검토 부탁드립니다",
     "body": "안녕하세요, 내일 10시 회의 자료입니다. 검토 후 의견 부탁드려요."},

    {"subject": "🎉 50% 특가! 오늘만 할인",
     "body": "지금 구매하세요! 한정 수량으로 진행되는 깜짝 세일입니다."},

    {"subject": "면접 일정 확정 안내",
     "body": "선생님, 다음 주 화요일 오후 2시 면접 일정 확정되었습니다. 회신 부탁드립니다."},

    {"subject": "회식 일정 안내",
     "body": "이번 주 금요일 저녁 7시 회식 있습니다. 참석 여부 알려주세요."},
]

for e in emails:
    res = chain_email.invoke({"subject": e["subject"], "body": e["body"]})
    reply_mark = "📬 답장 필요" if res.requires_reply else "📭 답장 불필요"
    icon = {"high": "🔴", "medium": "🟡", "low": "🟢"}[res.priority]
    print(f"{icon} [{res.category}] {res.suggested_action}")
    print(f"   제목: {e['subject']}")
    print(f"   {reply_mark}  |  {res.one_line}")
    print()

🔴 [업무] 즉시 답장
   제목: [긴급] 내일 오전 회의 자료 검토 부탁드립니다
   📬 답장 필요  |  내일 회의 자료 검토 요청

🟢 [광고] 삭제
   제목: 🎉 50% 특가! 오늘만 할인
   📭 답장 불필요  |  50% 할인 광고 메일

🔴 [업무] 즉시 답장
   제목: 면접 일정 확정 안내
   📬 답장 필요  |  면접 일정 확정 안내 및 회신 요청

🟡 [업무] 오늘 중 답장
   제목: 회식 일정 안내
   📬 답장 필요  |  회식 참석 여부 확인 요청



**👀 관찰 포인트**

- 광고 메일 → 자동으로 `requires_reply=False`, `suggested_action="삭제"`
- 회의 요청 → `requires_reply=True`, `priority="high"`
- `bool` 필드로 LLM의 판단을 **참/거짓 형태**로 받음

Gmail, Outlook 같은 메일 서비스가 내부적으로 사용하는 패턴입니다.


---

# Step 6. `batch()` 로 여러 건 동시 처리

지금까지는 `for` 루프로 한 건씩 호출했어요. 실무에서는 **수십·수백 건을 빠르게** 처리해야 합니다.

LangChain의 모든 체인은 `.batch()` 메서드로 여러 입력을 **자동 병렬 처리** 합니다. 코드를 거의 안 바꾸고 속도가 빨라져요.

## 6-1. for 루프 vs batch 비교


In [23]:
import time

# 테스트할 리뷰 5개
test_reviews = [
    {"review": "정말 좋은 제품이에요!"},
    {"review": "배송이 너무 느려요."},
    {"review": "가격 대비 만족합니다."},
    {"review": "포장이 부실했어요."},
    {"review": "디자인이 예뻐요."},
]

# 방법 A: for 루프 (순차 처리)
start = time.time()
results_seq = [chain_review.invoke(x) for x in test_reviews]
time_seq = time.time() - start
print(f"⏱️  for 루프: {time_seq:.2f}초")

# 방법 B: batch (병렬 처리)
start = time.time()
results_batch = chain_review.batch(test_reviews)
time_batch = time.time() - start
print(f"⚡ batch:    {time_batch:.2f}초")

print(f"\n📊 속도 향상: 약 {time_seq / time_batch:.1f}배")

⏱️  for 루프: 5.65초
⚡ batch:    1.62초

📊 속도 향상: 약 3.5배


**👀 결과**

`batch` 가 보통 2~3배 빠릅니다. 입력이 100건이면 차이가 훨씬 더 커져요.

코드 변화는 **딱 한 줄**:

```python
# 순차 (느림)
results = [chain.invoke(x) for x in inputs]

# 병렬 (빠름)
results = chain.batch(inputs)
```

## 6-2. batch 결과 활용

`batch` 결과도 Pydantic 객체 리스트라서 그대로 활용 가능합니다.


In [24]:
# 5건 결과를 한 번에 정리
for review, result in zip(test_reviews, results_batch):
    print(f"⭐ {result.rating}/5 | {result.sentiment:10s} | {review['review']}")

⭐ 5/5 | positive   | 정말 좋은 제품이에요!
⭐ 2/5 | negative   | 배송이 너무 느려요.
⭐ 4/5 | positive   | 가격 대비 만족합니다.
⭐ 2/5 | negative   | 포장이 부실했어요.
⭐ 5/5 | positive   | 디자인이 예뻐요.


---

# Step 7. 🎯 도전 과제

본인이 가장 관심 있는 사례 하나를 골라 깊이 시도해보세요. **새로운 스키마 + 같은 패턴**으로 만들면 됩니다.

## 과제 A: SNS 댓글 모더레이션 (실무 빈도 ★★★★★)

악성 댓글을 자동 필터링하는 시스템.

**스키마 힌트**
```python
class CommentModeration(BaseModel):
    toxicity: Literal["safe", "warning", "toxic"]
    contains_profanity: bool          # 욕설 포함 여부
    contains_personal_attack: bool    # 인신공격 여부
    reason: Optional[str]              # 문제가 있을 때 그 이유
```

테스트할 댓글 3-5개를 본인이 만들어서 분류해보세요.

---

## 과제 B: 식당 리뷰 → 메뉴별 평가 추출 (실무 빈도 ★★★★☆)

식당 리뷰 하나에 여러 메뉴 평가가 섞여있을 때, **메뉴마다 따로 평가**를 추출.

**스키마 힌트** — 중첩 모델
```python
class MenuRating(BaseModel):
    name: str
    rating: int = Field(ge=1, le=5)
    comment: str

class RestaurantReview(BaseModel):
    overall_rating: int = Field(ge=1, le=5)
    menus: list[MenuRating]    # 메뉴별 평가 리스트
    would_revisit: bool
```

**테스트 입력**: "파스타는 정말 맛있었는데 5점! 그런데 피자는 좀 짜서 3점이에요. 다시 갈 의향은 있어요."

---

## 과제 C: 일정 정보 추출 (실무 빈도 ★★★★☆)

자연어 메시지에서 **날짜·시간·장소** 를 추출해서 캘린더 등록용 정형 데이터로.

**스키마 힌트**
```python
class Meeting(BaseModel):
    title: str
    date: str           # "2025-12-15"
    time: str           # "14:00"
    location: Optional[str]
    attendees: list[str]
```

**테스트 입력**: "내일 오후 3시에 강남역 스타벅스에서 김민지 매니저랑 프로젝트 미팅"

---

## 과제 D: 본인이 자주 마주치는 비정형 데이터 (★★★)

업무·일상에서 자주 보지만 정형화되지 않은 데이터가 있다면 시도해보세요.

- 회의록 → 액션 아이템 추출
- 학생 자기소개서 → 장단점 키워드 추출
- 이메일 → 캘린더 일정 추출
- 카톡 단톡방 메시지 → 약속 정리

본인 도메인에 가까울수록 학습 효과가 큽니다.

---

> 💡 **하나만 골라서 깊이.** 5단계 패턴을 한 번 더 손으로 짜보는 게 핵심입니다.


---

# 🎓 1교시 마무리

## 오늘 배운 빌드업

```
[Day 1]         prompt | llm                          → AIMessage
[+ 추가 ①]     prompt | llm | StrOutputParser()      → str
[+ 추가 ②] ⭐  prompt | llm | PydanticOutputParser   → 객체!
```

체인 끝에 **파서 하나** 추가했을 뿐인데 LLM이 챗봇에서 **데이터 처리 시스템**으로 변신했습니다.

## 5단계 표준 패턴 (꼭 외우세요!)

```python
# 1. 스키마
class MySchema(BaseModel):
    field1: int = Field(description="...")
    field2: Literal["a", "b", "c"]

# 2. 파서
parser = PydanticOutputParser(pydantic_object=MySchema)

# 3. 프롬프트 (format_instructions 자동 삽입)
prompt = ChatPromptTemplate.from_messages([
    ("system", "...\n\n{format_instructions}"),
    ("human", "{input}")
]).partial(format_instructions=parser.get_format_instructions())

# 4. LLM (temperature=0)
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

# 5. 체인
chain = prompt | llm | parser
result = chain.invoke({"input": "..."})
```

오늘 만든 **5가지 케이스(리뷰·문의·이력서·뉴스·이메일)** 가 모두 이 5단계로 만들어졌습니다.

## 다음 시간 예고 (2교시)

다음 2교시는 **RunnableParallel** 입니다. 오늘 만든 분석 체인에 **답장 체인**을 추가해서 한 번 호출에 두 결과를 받는 법을 배워요.

```python
# 2교시 미리보기
full = RunnableParallel(
    analysis=chain_review,   # 오늘 만든 분석 체인
    reply=reply_chain        # 답장 체인 (내일)
)
result = full.invoke({"review": "..."})
result["analysis"]   # Pydantic 객체
result["reply"]      # 문자열 답장
```

수고하셨습니다! 🎉
